# 32 整数规划与目标规划

> **模块**：优化 ｜ **数据**：无（内置示例） ｜ **依赖**：numpy、pulp

## 1. 问题背景

整数规划要求部分或全部变量取整，目标规划则用偏差变量处理多目标冲突。两者都是线性规划的扩展，常见于排班、选址与多目标决策。

## 2. 学习目标

- 用 `pulp` 建立整数规划模型并求解
- 理解目标规划中正、负偏差变量的含义
- 能解读求解状态与灵敏度信息

## 3. 环境与数据

- 依赖：numpy、pulp
- 数据：无（内置示例）

## 4. 方法与实现

### 4.1 整数规划

In [ ]:
# Integer Program Problem
'''
here is an example to solve the following problem:
max z = x[0]**2 + x[1]**2 + 3*x[2]**2 + 4*x[3]**2 + 2*x[4]**2 - 8*x[0] - 2*x[1] - 3*x[2] - x[3] - 2*x[4]
s.t   x in bounds
      A_ub * x <= b_ub
here x is an array of integers.
'''

import numpy as np

A_ub = [[1, 1, 1, 1, 1],
        [1, 2, 2, 1, 6],
        [2, 1, 6, 0, 0],
        [0, 0, 1, 1, 5]]

b_ub = [400, 800, 200, 200]
bounds = [(0, 99), (0, 99), (0, 99), (0, 99)]

def z(x):
    return x[0]**2 + x[1]**2 + 3*x[2]**2 + 4*x[3]**2 + 2*x[4]**2 - 8*x[0] - 2*x[1] - 3*x[2] - x[3] - 2*x[4]

In [ ]:
# Monte Carlo Method 蒙特卡洛方法

x0 = []
z0 = 0
for i in range(int(1000000)):
    x = np.random.randint(low=0, high=99, size=5, dtype=np.int64)
    if(np.max(np.matmul(A_ub, x)-b_ub) > 0):
        continue
    tempz = z(x)
    if tempz > z0:
        z0 = tempz
        x0 = x
        
print('The best z is :', z0)
print('The according x is: ', x0)

In [ ]:
from pulp import *

# 1. 建立问题
prob = LpProblem("Bleding Problem", LpMinimize)

# 2. 建立变量
x1 = LpVariable("ChickenPercent", 0, None, LpInteger)
x2 = LpVariable("BeefPercent", 0)

# 3. 设置目标函数
prob += 0.013*x1 + 0.008*x2, "Total Cost of Ingredients per can"

# 4. 施加约束
prob += x1 + x2 == 100, "PercentagesSum"
prob += 0.100*x1 + 0.200*x2 >= 8.0, "ProteinRequirement"
prob += 0.080*x1 + 0.100*x2 >= 6.0, "FatRequirement"
prob += 0.001*x1 + 0.005*x2 <= 2.0, "FibreRequirement"
prob += 0.002*x2 + 0.005*x2 <= 0.4, "SaltRequirement"

# 5. 求解
prob.solve()

# 6. 打印求解状态
print("Status:", LpStatus[prob.status])

# 7. 打印出每个变量的最优值
for v in prob.variables():
    print(v.name, "=", v.varValue)

# 8. 打印最优解的目标函数值
print("Total Cost of Ingredients per can = ", value(prob.objective))

In [ ]:
# 例2.8

from pulp import *

# 1. 建立问题
prob = LpProblem("MIP", LpMinimize)

# 2. 建立变量
x = LpVariable.dicts(name="x", indexs=[1, 2], lowBound=0, cat='Continuous')
x3 = LpVariable(name='x3', lowBound=0, upBound=1, cat='Integer')
x[2].setInitialValue(5.5)
x3.setInitialValue(1)

# 3. 设置目标函数
prob += -3*x[1]-2*x[2]-x3

# 4. 施加约束
prob += x[1] + x[2] + x3 <= 7
prob += 4*x[1] + 2*x[2] + x3 == 12

# 5. 求解
prob.solve()

# 6. 打印求解状态
print("Status:", LpStatus[prob.status])

# 7. 打印出每个变量的最优值
for v in prob.variables():
    print(v.name, "=", v.varValue)

# 8. 打印最优解的目标函数值
print("Total Cost of Ingredients per can = ", value(prob.objective))

### 4.2 目标规划

**基本上分为两种思路: 加权系数法(转化为单一目标), 优先等级法(按重要程度不同, 转化为单目标模型)**

正负偏差变量: $d_i^+=max\{f_i-d_i^0,0\},\ d_i^-=-min\{f_i-d_i^0,0\}$, 显然恒有$d_i^+\times d_i^-=0$

In [ ]:
# 例16.3
# 该题的规划模型如下:
# min z =P[0]*dminus[0] + P[1]*(dplus[1]+dminus[1]) + P[2]*(3 * (dplus[2]+dminus[2]) + dplus[3])
#     2*x[0] + 2*x[1] <= 12
#     200*x[0] + 300*x[1] + dminus[0] - dplus[0] = 1500
#     2*x[0] - x[1] + dminus[1] - dplus[1] = 0
#     4*x[0] + dminus[2] - dplus[2] = 16
#     5*x[1] + dminus[3] - dplus[3] = 15
#     x[0], x[1], dminus[i], dplus[i] >= 0, i = 0, 1, 2, 3


## 5. 小结

`pulp` 建模按“变量—目标—约束—求解”四步进行；目标规划把多个目标写成偏差之和，再最小化总偏差。

> 内容来源：教材参考实现《数学建模算法与应用》练习稿（MIT，Zhanghan Wang），原始文件已并入本篇，见 `THIRD-PARTY.md`。